In [11]:
# 01 数据探索
# 淘宝用户行为数据集 · 初探

import pandas as pd
import numpy as np

# 设置显示选项
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

In [16]:
COLUMNS = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']

df = pd.read_csv('../data/raw/UserBehavior.csv', 
                 names=COLUMNS,
                 nrows=5000000)

print(f"行数: {len(df):,}")
print(f"列名: {list(df.columns)}")
print(f"\n前5行:")
df.head()

行数: 5,000,000
列名: ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']

前5行:


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [17]:
# 检查缺失值
print("缺失值统计:")
print(df.isnull().sum())
print(f"\n缺失值占比: {df.isnull().sum().sum() / len(df) * 100:.2f}%")

缺失值统计:
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64

缺失值占比: 0.00%


In [18]:
# 检查重复值
dup = df.duplicated().sum()
print(f"完全重复行: {dup:,} ({dup/len(df)*100:.2f}%)")

完全重复行: 5 (0.00%)


In [19]:
# 行为类型分布
behavior_counts = df['behavior_type'].value_counts()
print("行为类型分布:")
for b, c in behavior_counts.items():
    print(f"  {b}: {c:,} ({c/len(df)*100:.1f}%)")
print(f"\n整体转化率 (pv → buy): {behavior_counts.get('buy',0) / behavior_counts.get('pv',1) * 100:.2f}%")

行为类型分布:
  pv: 4,475,232 (89.5%)
  cart: 279,512 (5.6%)
  fav: 145,125 (2.9%)
  buy: 100,131 (2.0%)

整体转化率 (pv → buy): 2.24%


In [20]:
# 去重统计
print(f"用户数: {df['user_id'].nunique():,}")
print(f"商品数: {df['item_id'].nunique():,}")
print(f"类目数: {df['category_id'].nunique():,}")
print(f"人均行为数: {len(df) / df['user_id'].nunique():.1f}")

用户数: 48,984
商品数: 1,080,623
类目数: 7,354
人均行为数: 102.1


In [21]:
# 时间戳处理
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"覆盖天数: {(df['timestamp'].max() - df['timestamp'].min()).days} 天")
print(f"数据分布:")
print(df['timestamp'].dt.date.value_counts().sort_index().head(10))

时间范围: 1970-01-01 12:13:36 ~ 2037-04-09 05:22:35
覆盖天数: 24569 天
数据分布:
timestamp
1970-01-01    1
2015-02-06    1
2017-07-03    2
2017-09-11    1
2017-09-15    1
2017-09-16    1
2017-10-07    1
2017-10-10    1
2017-10-31    1
2017-11-01    1
Name: count, dtype: int64


In [23]:
# 前面已经转成 datetime 了，直接过滤
df = df[(df['timestamp'] >= '2017-01-01') & (df['timestamp'] <= '2018-12-31')]
print(f"过滤后行数: {len(df):,}")
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"覆盖天数: {(df['timestamp'].max() - df['timestamp'].min()).days} 天")
print(f"\n每日记录数:")
print(df['timestamp'].dt.date.value_counts().sort_index())

过滤后行数: 4,999,993
时间范围: 2017-07-03 09:24:32 ~ 2018-08-28 10:27:12
覆盖天数: 421 天

每日记录数:
timestamp
2017-07-03         2
2017-09-11         1
2017-09-15         1
2017-09-16         1
2017-10-07         1
2017-10-10         1
2017-10-31         1
2017-11-01         1
2017-11-02         2
2017-11-03        70
2017-11-05         1
2017-11-06         1
2017-11-10         3
2017-11-11         8
2017-11-12         8
2017-11-13         3
2017-11-14         5
2017-11-15         4
2017-11-16        12
2017-11-17        23
2017-11-18        21
2017-11-19        39
2017-11-20        42
2017-11-21        39
2017-11-22        90
2017-11-23       264
2017-11-24     59861
2017-11-25    518658
2017-11-26    525482
2017-11-27    498957
2017-11-28    490209
2017-11-29    513578
2017-11-30    524675
2017-12-01    561057
2017-12-02    703085
2017-12-03    603768
2017-12-06         3
2018-08-28        16
Name: count, dtype: int64


In [24]:
# 1.4 数据清洗
# ① 只保留核心时间段（11-24 到 12-03）
df = df[(df['timestamp'] >= '2017-11-24') & (df['timestamp'] <= '2017-12-04')]
print(f"过滤后行数: {len(df):,}")


过滤后行数: 4,999,330


In [25]:
# ② 去重
df = df.drop_duplicates()


In [26]:
# ③ 提取日期和小时（方便后续分析）
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

print(f"最终数据量: {len(df):,} 行, {df['user_id'].nunique():,} 用户")
print(f"日期范围: {df['date'].min()} ~ {df['date'].max()}")

最终数据量: 4,999,325 行, 48,984 用户
日期范围: 2017-11-24 ~ 2017-12-03


In [27]:
# ④ 存入 DuckDB
import duckdb
con = duckdb.connect('../data/processed/ecommerce.db')
con.execute("DROP TABLE IF EXISTS behaviors")
con.execute("CREATE TABLE behaviors AS SELECT * FROM df")
print(f"\nDuckDB 表行数: {con.execute('SELECT COUNT(*) FROM behaviors').fetchone()[0]:,}")
con.close()
print("数据已存入 data/processed/ecommerce.db ✅")


DuckDB 表行数: 4,999,325
数据已存入 data/processed/ecommerce.db ✅
